# Feature Engineering


### 1. Import Data Processing Libraries
Loads numerical and data processing tools (pandas, numpy, os).


In [8]:
import os
import pandas as pd
import numpy as np

print('Feature engineering environment initialized.')

Feature engineering environment initialized.


### 2. Load Cleaned Transactions
Loads data/processed/cleaned_transactions.csv created in 01_data_cleaning.ipynb.


In [9]:
data_path = os.path.join('..', 'data', 'processed', 'cleaned_transactions.csv')
df = pd.read_csv(data_path)
df['timestamp'] = pd.to_datetime(df['timestamp'])
print(f'Transaction records loaded: {len(df):,} rows x {df.shape[1]} columns')
df[['transaction_id', 'account_id', 'amount', 'amount_to_avg_ratio', 'avg_transaction_amount_30d', 'is_fraud']].head(3)

Transaction records loaded: 126,000 rows x 29 columns


,transaction_id,account_id,amount,amount_to_avg_ratio,avg_transaction_amount_30d,is_fraud
0,FLR250216186265,FLR-ACC-100000,84233.00,9.21,9143.33,0
1,FLR250423143305,FLR-ACC-100000,9143.33,1.00,9143.33,0
2,FLR250424154897,FLR-ACC-100000,20639.00,2.26,9143.33,0


### 3. Smooth Out Extreme Dollar Amounts with Log Scaling
Payment amounts range from a few dollars all the way into millions of dollars. Huge numbers can overwhelm simple models and skew calculations. A standard math technique called a logarithm (log) shrinks the scale and pulls extreme values closer together without changing their relative order.


In [10]:
# Compute absolute amount and log transformations
df['abs_amount'] = df['amount'].abs()
df['log_amount'] = np.log1p(df['abs_amount'])
df['log_amount_to_avg_ratio'] = np.log1p(np.maximum(df['amount_to_avg_ratio'], 0))
df['log_avg_30d'] = np.log1p(np.maximum(df['avg_transaction_amount_30d'], 0))

print('Transformed Feature Summary Statistics')
print(df[['log_amount', 'log_amount_to_avg_ratio', 'log_avg_30d']].describe().round(3))
df[['amount', 'log_amount', 'amount_to_avg_ratio', 'log_amount_to_avg_ratio']].head(4)

Transformed Feature Summary Statistics
       log_amount  log_amount_to_avg_ratio  log_avg_30d
count  126000.000               126000.000   126000.000
mean        8.281                    0.835        8.595
std         4.318                    0.910        4.222
min         0.000                    0.000        0.000
25%         4.176                    0.223        4.537
50%         8.710                    0.683        8.690
75%        11.705                    1.040       11.968
max        19.411                   10.366       18.971


,amount,log_amount,amount_to_avg_ratio,log_amount_to_avg_ratio
0,84233.00,11.341354,9.21,2.323368
1,9143.33,9.120889,1.00,0.693147
2,20639.00,9.934986,2.26,1.181727
3,3177.43,8.064143,0.21,0.190620


### 4. Extract Device, Velocity, and Location Signals
Gathers key behavioral clues: how many payments occurred in the last hour (transaction_velocity_1h), whether the device is new (is_new_device), whether it is an international payment (is_cross_border), and whether it was a refund (is_negative_amount).


In [11]:
df['is_new_device'] = df['is_new_device'].fillna(0).astype(int)
df['is_negative_amount'] = (df['amount'] < 0).astype(int)

behavioral_cols = ['transaction_velocity_1h', 'is_new_device', 'is_cross_border', 'is_negative_amount']
print('Behavioral signals summary:')
print(df[behavioral_cols].describe().round(3))

Behavioral signals summary:
       transaction_velocity_1h  is_new_device  is_cross_border  \
count               126000.000     126000.000       126000.000   
mean                     0.011          0.031            0.020   
std                      0.110          0.173            0.141   
min                      0.000          0.000            0.000   
25%                      0.000          0.000            0.000   
50%                      0.000          0.000            0.000   
75%                      0.000          0.000            0.000   
max                      3.000          1.000            1.000   

       is_negative_amount  
count          126000.000  
mean                0.002  
std                 0.048  
min                 0.000  
25%                 0.000  
50%                 0.000  
75%                 0.000  
max                 1.000  


### 5. Convert Category Labels into Numerical Indicators
Computers understand numbers, not text labels. Categories (like account type, verification level, channel, and store type) are converted into simple 1 or 0 (yes/no) columns.


In [12]:
categorical_cols = ['account_type', 'kyc_tier', 'channel', 'merchant_category']
encoded_df = pd.get_dummies(df, columns=categorical_cols, drop_first=True, dtype=int)

dummy_cols = [c for c in encoded_df.columns if any(c.startswith(cat + '_') for cat in categorical_cols)]
print(f'Generated {len(dummy_cols)} dummy features across categories.')
print('Sample dummy columns:', dummy_cols[:6])

Generated 22 dummy features across categories.
Sample dummy columns: ['account_type_Individual', 'kyc_tier_Tier2_Verified', 'kyc_tier_Tier3_Enhanced', 'channel_Card Not Present', 'channel_Card Present', 'channel_Mobile App']


### 6. Check for Data Leakage and Remove Sensitive Identifiers
Removes customer names, account IDs, descriptions, and timestamps. This ensures the model learns true behavioral habits rather than memorizing individual people or peeking at information it would not have during a live transaction.


In [13]:
# Exclude all non-feature metadata and post-settlement indicators
exclude_cols = [
    'transaction_id', 'account_id', 'timestamp', 'day_of_week', 'description', 'merchant_name',
    'currency', 'transaction_country', 'home_country', 'device_id', 'status',
    'account_holder_name', 'account_created_date', 'amount', 'abs_amount',
    'personal_spend_baseline_usd', 'is_fraud'
]
model_features = [col for col in encoded_df.columns if col not in exclude_cols]

print(f'Total candidate predictor features: {len(model_features)}')
print('Selected feature names:\n', model_features)

Total candidate predictor features: 33
Selected feature names:
 ['hour_of_day', 'amount_to_avg_ratio', 'avg_transaction_amount_30d', 'transaction_velocity_1h', 'is_cross_border', 'is_new_device', 'account_age_days', 'is_negative_amount', 'log_amount', 'log_amount_to_avg_ratio', 'log_avg_30d', 'account_type_Individual', 'kyc_tier_Tier2_Verified', 'kyc_tier_Tier3_Enhanced', 'channel_Card Not Present', 'channel_Card Present', 'channel_Mobile App', 'channel_USSD', 'channel_Web Dashboard', 'merchant_category_Crypto Exchange', 'merchant_category_Electronics', 'merchant_category_Gambling/Gaming', 'merchant_category_Groceries', 'merchant_category_Healthcare', 'merchant_category_Insurance', 'merchant_category_P2P Transfer', 'merchant_category_Payroll Transfer', 'merchant_category_Restaurants', 'merchant_category_Retail', 'merchant_category_Subscription/SaaS', 'merchant_category_Travel', 'merchant_category_Utilities', 'merchant_category_Wire Transfer']


### 7. Save the Engineered Feature Matrix
Saves the complete list of model clues into data/processed/feature_matrix.csv.


In [14]:
proc_dir = os.path.join('..', 'data', 'processed')
matrix_path = os.path.join(proc_dir, 'feature_matrix.csv')

encoded_df.to_csv(matrix_path, index=False)
print(f'Feature matrix successfully saved to: {matrix_path}')
print(f'Dimensions: {encoded_df.shape[0]:,} rows x {encoded_df.shape[1]} columns')

Feature matrix successfully saved to: ..\data\processed\feature_matrix.csv
Dimensions: 126,000 rows x 50 columns


Summary of Features Prepared:
- log_amount: The payment size, scaled down smoothly so huge values do not distort the model.
- log_amount_to_avg_ratio: How much higher this payment is compared to the customer's normal spending.
- log_avg_30d: The customer's typical spending baseline.
- transaction_velocity_1h: How many payments happened in the past hour (rapid bursts).
- is_new_device: 1 if sent from an unfamiliar phone or computer, 0 if familiar.
- is_cross_border: 1 if international, 0 if local.
- is_negative_amount: 1 if a refund or reversed payment.
- Category flags: Simple yes/no (1/0) flags for account types, verification levels, channels, and store categories.

---
Next Step: In 04_model_development.ipynb, these clues are used to train and compare two different fraud detection models.
